In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Definition of parameters
L = 10  # Signal length
n_signal = np.arange(L)
x = np.exp(-0.1 * n_signal) * np.cos(2 * np.pi * 0.1 * n_signal)

# 2. Interactive plotting function for the Complex Z-Plane
def plot_z_plane_correlation(N, omega_idx):
    # DFT samples frequencies
    k_indices = np.arange(N)
    omega_k = 2 * np.pi * k_indices / N
    
    # DFT coefficients X[k] for the signal x[n]
    xi_n = np.zeros(N)
    for n in range(N):
        for m in range(-3, 4):
            idx = n - m * N
            if 0 <= idx < L:
                xi_n[n] += x[idx]
    X_k = np.fft.fft(xi_n, N)
    
    # Selected frequency on the unit circle
    omega_val = np.linspace(0, 2 * np.pi, 200)[omega_idx]
    z_real = np.cos(omega_val)
    z_imag = np.sin(omega_val)
    
    # Setup figure
    fig, ax = plt.subplots(figsize=(7, 7))
    
    # Draw Unit Circle
    theta = np.linspace(0, 2 * np.pi, 500)
    ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.5, label=r'Unit Circle ($|z|=1$)')
    
    # Draw DFT samples on the unit circle
    dft_real = np.cos(omega_k)
    dft_imag = np.sin(omega_k)
    ax.scatter(dft_real, dft_imag, color='red', s=80, zorder=5, label=rf'DFT Samples $\mathcal{{X}}[k]$ ($N={N}$)')
    
    # Draw lines connecting the evaluated point $z = e^{j\omega}$ to all DFT sample roots
    for i in range(N):
        ax.plot([z_real, dft_real[i]], [z_imag, dft_imag[i]], color='gray', linestyle=':', alpha=0.7)

    # Draw evaluated point $z = e^{j\omega}$
    ax.scatter([z_real], [z_imag], color='blue', s=120, zorder=6, marker='s', label=rf'Evaluated $z = e^{{j\omega}}$ ($\omega$={omega_val:.2f} rad)')
    
    # Axis formatting
    ax.axhline(0, color='black', linewidth=0.8)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlim([-1.4, 1.4])
    ax.set_ylim([-1.4, 1.4])
    ax.set_aspect('equal')
    ax.set_title(rf'Z-Plane Geometry: Connecting DFT Samples to $z = e^{{j\omega}}$ ($N = {N}$)', fontsize=11, fontweight='bold')
    ax.set_xlabel(r'Real Part ($\sigma$)', fontsize=10)
    ax.set_ylabel(r'Imaginary Part ($j\Omega$)', fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(loc='upper right', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# 3. Explanatory text and enriched geometric conclusion widget
instructions_html = HTML(r"""
<div style="background-color: #f8f9fa; padding: 12px; border-left: 4px solid #28a745; margin-bottom: 10px; font-family: Arial, sans-serif; font-size: 13px; line-height: 1.5;">
    <b>Interactive Exploration: DFT and $\mathcal{Z}$-Transform Relationship in the Complex Plane</b><br>
    <ul>
        <li><b>Red Dots:</b> The $N$ equidistant DFT sample points $\exp\left(j\frac{2\pi k}{N}\right)$ lying on the unit circle.</li>
        <li><b>Blue Square:</b> The evaluation frequency point $z = e^{j\omega}$ on the unit circle.</li>
        <li><b>Gray Dotted Lines:</b> Represent the denominator factors $\left(1 - z^{-1}e^{j\frac{2\pi k}{N}}\right)$ from the interpolation formula.</li>
    </ul>
    <b>Geometric Conclusion:</b> To compute the spectrum at any arbitrary point $z = e^{j\omega}$ (blue marker), <b>all DFT samples (red dots) participate simultaneously</b>. The lengths of the connecting gray lines act directly as <b>weighting factors</b> (inversely proportional via the denominator): when the evaluation point approaches a specific DFT sample, that sample's contribution dominates due to the minimal distance, blending all surrounding bins into the final continuous frequency response.
</div>
""")
display(instructions_html)

# 4. Widgets for Interactivity
slider_N = widgets.IntSlider(value=8, min=4, max=24, step=1, description='N (DFT Size):', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))
slider_omega = widgets.IntSlider(value=25, min=0, max=199, step=1, description=r'Frequency $\omega$:', style={'description_width': 'initial'}, layout=widgets.Layout(width='400px'))

widgets.interactive(plot_z_plane_correlation, N=slider_N, omega_idx=slider_omega)